[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/01_fundamentals/optimizers_losses.ipynb)



# 05 . Optimizers and Losses

## Concept — Optimizers & Losses (revision notes)

**What an optimizer is**
- An algorithm that updates model parameters using their gradients, aiming
  to minimize the loss.
- Every optimizer in `torch.optim` implements the same interface:
  `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`.
- The optimizer holds references to the parameters it manages (passed in at
  construction) and reads their `.grad` to decide how to update them.

**Why plain SGD isn't always enough**
- Vanilla SGD: `param = param - lr * grad`. Simple, but slow to converge and
  sensitive to the learning rate choice.
- Real training almost always uses a variant that adapts the effective
  learning rate per-parameter or adds momentum.

**SGD with momentum**
- Adds a "velocity" term that accumulates a moving average of past
  gradients, so updates keep some direction from previous steps instead of
  reacting only to the current gradient.
- Helps escape shallow local minima / plateaus and dampens oscillation in
  narrow valleys of the loss surface.
- `torch.optim.SGD(params, lr=..., momentum=0.9)` — 0.9 is a very common default.

**Adam / AdamW — the modern default**
- Adam keeps a per-parameter moving average of BOTH the gradient (first
  moment, like momentum) and the squared gradient (second moment), and uses
  both to adapt the effective learning rate per parameter.
- Adam is often described as "less sensitive to the learning rate" than SGD
  because it adapts per-parameter step sizes using gradient history — this
  holds in many real, noisy, high-dimensional training settings, but it's
  not a universal law: on small, well-conditioned, noise-free problems,
  plain SGD can converge just as well or faster (see section 3 for a
  concrete, measured example of this). The practical takeaway: **tune the
  learning rate separately per optimizer** rather than assuming one LR value
  transfers between SGD and Adam.
- **AdamW** fixes a subtle bug in how the original Adam implements weight
  decay (L2 regularization) — it decouples weight decay from the gradient
  update instead of folding it into the gradient. **Prefer AdamW over Adam**
  whenever you're using weight decay (which is almost always).

**Key optimizer hyperparameters, quick reference**
- `lr` (learning rate) — the single most important hyperparameter; too high
  diverges, too low trains painfully slowly.
- `momentum` (SGD) — how much of the previous update direction to keep.
- `betas` (Adam/AdamW) — decay rates for the first and second moment
  estimates, default `(0.9, 0.999)`, rarely changed.
- `weight_decay` — L2-style regularization strength; shrinks weights toward
  zero each step to reduce overfitting.
- `eps` — tiny constant added for numerical stability (avoids divide-by-zero
  in adaptive methods); rarely changed from its default.

**Loss functions — what they measure and when to use them**
- `nn.MSELoss` — mean squared error; standard for regression (continuous targets).
- `nn.CrossEntropyLoss` — for multi-class classification. **Important:**
  expects raw logits (NOT softmax probabilities) and integer class labels
  (NOT one-hot vectors) — it applies `log_softmax` + `NLLLoss` internally.
- `nn.BCELoss` — binary classification, expects the model output to already
  be a probability (post-sigmoid) in `[0, 1]`.
- `nn.BCEWithLogitsLoss` — binary classification, expects raw logits (NOT
  post-sigmoid). Numerically more stable than `BCELoss` + separate sigmoid —
  **prefer this over manually applying sigmoid then BCELoss.**
- `nn.L1Loss` — mean absolute error; more robust to outliers than MSE.
- `nn.SmoothL1Loss` (Huber loss) — behaves like L2 for small errors and L1
  for large errors; a common middle ground for regression with outliers.

**The optimizer/loss/model relationship in one line**
- `loss = loss_fn(model(x), target)` computes a scalar.
  `loss.backward()` computes gradients for every parameter that contributed.
  `optimizer.step()` updates those parameters using the gradients and the
  optimizer's internal rule (SGD, Adam, etc.).

**Learning rate schedulers**
- Adjust the learning rate over the course of training instead of keeping it fixed.
- `StepLR` — multiply LR by a factor every N epochs (a "staircase" decay).
- `CosineAnnealingLR` — smoothly decay LR following a cosine curve — very
  popular for modern training recipes.
- `ReduceLROnPlateau` — reduce LR automatically when a monitored metric
  (usually validation loss) stops improving.
- `OneCycleLR` — ramps LR up then back down within a single training run,
  often paired with a higher peak LR than you'd otherwise dare use.
- Scheduler `.step()` is called once per epoch (most schedulers) or once per
  batch (`OneCycleLR`, some others) — always check the specific scheduler's
  docs/convention.

**Gradient clipping**
- `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)` rescales
  all gradients so their combined norm doesn't exceed `max_norm`.
- Call it AFTER `loss.backward()` and BEFORE `optimizer.step()`.
- Standard defense against exploding gradients, especially common in RNNs
  and early transformer training.


### 1. The optimizer/loss/backward/step cycle, end to end

**What this cell does (plain language):** we build the smallest possible
training loop — one linear layer learning to fit `y = 2x + 1` — so you can
see the four-line cycle (`zero_grad` → forward + loss → `backward` →
`step`) that every PyTorch training loop is built from, and watch the loss
actually go down over iterations.


In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])  # true relationship: y = 2x + 1

for step in range(100):
    optimizer.zero_grad()        # 1. clear old gradients
    pred = model(x)              # 2. forward pass
    loss = loss_fn(pred, y)      # 2. compute loss
    loss.backward()              # 3. compute gradients
    optimizer.step()             # 4. update parameters using those gradients

    if step % 20 == 0:
        print(f"step {step}: loss = {loss.item():.4f}")

print("\nlearned weight (expected ~2.0):", model.weight.item())
print("learned bias (expected ~1.0):", model.bias.item())


step 0: loss = 35.0928
step 20: loss = 0.0019
step 40: loss = 0.0006
step 60: loss = 0.0002
step 80: loss = 0.0000

learned weight (expected ~2.0): 1.996801733970642
learned bias (expected ~1.0): 1.0094032287597656


### 2. SGD vs SGD+momentum — momentum's overshoot behavior

**What this cell does (plain language):** we train the SAME tiny model
twice, once with plain SGD and once with SGD + momentum, both starting from
identical weights, and compare their loss curves. This particular toy
problem is small, convex, and noise-free, which is exactly where momentum's
usual advantage often does NOT show up plainly — watch for momentum actually
overshooting past the minimum before settling back down.


In [2]:
def make_model_and_data():
    torch.manual_seed(42)
    model = nn.Linear(1, 1)
    x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
    y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])
    return model, x, y

def train(optimizer_fn, steps=50):
    model, x, y = make_model_and_data()
    optimizer = optimizer_fn(model.parameters())
    loss_fn = nn.MSELoss()
    losses = []
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

plain_sgd_losses = train(lambda p: torch.optim.SGD(p, lr=0.01))
momentum_sgd_losses = train(lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9))

print("plain SGD    - loss at step 10:", round(plain_sgd_losses[10], 4), "| step 49:", round(plain_sgd_losses[49], 4))
print("SGD+momentum - loss at step 10:", round(momentum_sgd_losses[10], 4), "| step 49:", round(momentum_sgd_losses[49], 4))
print("\nNotice momentum's loss is HIGHER at step 10 - it overshoots past the")
print("minimum due to its accumulated velocity, before settling back down. By")
print("step 49 the two may end up close to each other, or momentum may even")
print("edge slightly ahead once it stops oscillating - the overshoot pattern")
print("early on is the important, reliable thing to notice, not which one")
print("wins by the last step (that detail is sensitive to lr/steps chosen).")
print("Momentum's benefit is much clearer on noisier (mini-batch) or")
print("ill-conditioned (elongated valley) loss surfaces, which is the more")
print("typical situation in real training - not on tiny, deterministic,")
print("already-easy toy problems like this one.")


plain SGD    - loss at step 10: 0.3318 | step 49: 0.0063
SGD+momentum - loss at step 10: 0.7798 | step 49: 0.0036

Notice momentum's loss is HIGHER at step 10 - it overshoots past the
minimum due to its accumulated velocity, before settling back down. By
step 49 the two may end up close to each other, or momentum may even
edge slightly ahead once it stops oscillating - the overshoot pattern
early on is the important, reliable thing to notice, not which one
wins by the last step (that detail is sensitive to lr/steps chosen).
Momentum's benefit is much clearer on noisier (mini-batch) or
ill-conditioned (elongated valley) loss surfaces, which is the more
typical situation in real training - not on tiny, deterministic,
already-easy toy problems like this one.


### 3. Adam vs SGD — same nominal learning rate means different things

**What this cell does (plain language):** we train the same tiny model with
plain SGD vs Adam using the SAME nominal learning rate, to demonstrate an
important, often-missed subtlety: Adam and SGD interpret "learning rate"
very differently (Adam normalizes step size using gradient history), so the
same numeric `lr` value does NOT produce comparable step sizes between them.


In [3]:
sgd_losses = train(lambda p: torch.optim.SGD(p, lr=0.01))
adam_losses = train(lambda p: torch.optim.Adam(p, lr=0.01))

print("SGD  lr=0.01 - loss at step 10:", round(sgd_losses[10], 4), "| step 49:", round(sgd_losses[49], 4))
print("Adam lr=0.01 - loss at step 10:", round(adam_losses[10], 4), "| step 49:", round(adam_losses[49], 4))
print("\nOn THIS problem, Adam is actually SLOWER at the same nominal lr - a")
print("useful correction to the common assumption that 'Adam is just always faster'.")
print("Adam normalizes each parameter's step using a running estimate of its own")
print("gradient magnitude, so its effective step size behaves very differently")
print("from SGD's raw (lr * gradient) step. Practically: Adam and SGD usually")
print("need DIFFERENT learning rates tuned separately - you cannot assume a")
print("value that works for one will work similarly for the other.")

adam_tuned_losses = train(lambda p: torch.optim.Adam(p, lr=0.1))
print("\nAdam with a properly-tuned lr=0.1 instead:")
print("Adam lr=0.1  - loss at step 10:", round(adam_tuned_losses[10], 4), "| step 49:", round(adam_tuned_losses[49], 4))
print("Now Adam converges quickly too - the lesson is TUNE PER OPTIMIZER, not")
print("assume one lr value transfers between optimizer choices.")


SGD  lr=0.01 - loss at step 10: 0.3318 | step 49: 0.0063
Adam lr=0.01 - loss at step 10: 10.0803 | step 49: 3.5595

On THIS problem, Adam is actually SLOWER at the same nominal lr - a
useful correction to the common assumption that 'Adam is just always faster'.
Adam normalizes each parameter's step using a running estimate of its own
gradient magnitude, so its effective step size behaves very differently
from SGD's raw (lr * gradient) step. Practically: Adam and SGD usually
need DIFFERENT learning rates tuned separately - you cannot assume a
value that works for one will work similarly for the other.

Adam with a properly-tuned lr=0.1 instead:
Adam lr=0.1  - loss at step 10: 0.1273 | step 49: 0.0483
Now Adam converges quickly too - the lesson is TUNE PER OPTIMIZER, not
assume one lr value transfers between optimizer choices.


### 4. Adam vs AdamW — weight decay done correctly

**What this cell does (plain language):** we show the actual difference
between `Adam(weight_decay=...)` and `AdamW(weight_decay=...)` by inspecting
their parameter update formulas conceptually, and training both briefly to
show they behave slightly differently even with identical settings, because
AdamW applies weight decay directly to the weights rather than folding it
into the gradient.


In [4]:
torch.manual_seed(0)
model_adam = nn.Linear(1, 1)
torch.manual_seed(0)
model_adamw = nn.Linear(1, 1)

# confirm they start identical
print("identical starting weights:", torch.allclose(model_adam.weight, model_adamw.weight))

opt_adam = torch.optim.Adam(model_adam.parameters(), lr=0.1, weight_decay=0.1)
opt_adamw = torch.optim.AdamW(model_adamw.parameters(), lr=0.1, weight_decay=0.1)

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])
loss_fn = nn.MSELoss()

for _ in range(20):
    for model, opt in [(model_adam, opt_adam), (model_adamw, opt_adamw)]:
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()

print("\nafter 20 steps with weight_decay=0.1:")
print("Adam  weight:", model_adam.weight.item())
print("AdamW weight:", model_adamw.weight.item())
print("(they diverge because Adam folds weight decay into the gradient/moment")
print(" estimates, while AdamW subtracts it directly from the weights each step)")


identical starting weights: True

after 20 steps with weight_decay=0.1:
Adam  weight: 1.67351496219635
AdamW weight: 1.5503606796264648
(they diverge because Adam folds weight decay into the gradient/moment
 estimates, while AdamW subtracts it directly from the weights each step)


### 5. CrossEntropyLoss — logits in, integer labels in

**What this cell does (plain language):** we build a tiny 3-class
classification example and show exactly what `nn.CrossEntropyLoss` expects
as input — RAW logits (not softmax output) and integer class indices (not
one-hot vectors) — since getting this wrong is one of the most common
classification bugs.


In [5]:
import torch.nn.functional as F

logits = torch.tensor([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]])  # shape (batch=2, classes=3), RAW logits
labels = torch.tensor([0, 1])  # integer class index per sample, NOT one-hot

loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)
print("CrossEntropyLoss (raw logits + integer labels):", loss.item())

# Common bug: applying softmax yourself first, then passing that to CrossEntropyLoss
# This DOUBLE-applies the softmax internally and gives the WRONG loss value
softmaxed = F.softmax(logits, dim=1)
wrong_loss = loss_fn(softmaxed, labels)
print("WRONG - passing already-softmaxed input:", wrong_loss.item(), "<- different (and incorrect) value")

# If you want the actual predicted probabilities for inspection, do it separately:
probs = F.softmax(logits, dim=1)
print("\npredicted probabilities (for your own inspection only, not for the loss):\n", probs)


CrossEntropyLoss (raw logits + integer labels): 0.31853973865509033
WRONG - passing already-softmaxed input: 0.745010256767273 <- different (and incorrect) value

predicted probabilities (for your own inspection only, not for the loss):
 tensor([[0.6590, 0.2424, 0.0986],
        [0.1086, 0.8025, 0.0889]])


### 6. BCEWithLogitsLoss vs BCELoss — numerical stability

**What this cell does (plain language):** we compare `nn.BCELoss` (needs a
sigmoid applied manually first) against `nn.BCEWithLogitsLoss` (takes raw
logits directly) on the same data, showing both give the same result in the
normal case, then show why `BCEWithLogitsLoss` is the safer default with an
extreme logit value that would otherwise risk numerical issues.


In [ ]:
logits = torch.tensor([2.0, -1.5, 0.3])
targets = torch.tensor([1.0, 0.0, 1.0])

# Method 1: manual sigmoid + BCELoss
probs = torch.sigmoid(logits)
bce_loss = nn.BCELoss()(probs, targets)
print("BCELoss (manual sigmoid first):", bce_loss.item())

# Method 2: BCEWithLogitsLoss directly on raw logits - preferred
bce_logits_loss = nn.BCEWithLogitsLoss()(logits, targets)
print("BCEWithLogitsLoss (raw logits):", bce_logits_loss.item())
print("same result in the normal case:", torch.allclose(bce_loss, bce_logits_loss, atol=1e-6))

# Now with an extreme logit value - manual sigmoid can saturate to exactly
# 0.0 or 1.0 in floating point, making log(0) = -inf inside BCELoss
extreme_logits = torch.tensor([-100.0])
extreme_target = torch.tensor([1.0])
extreme_probs = torch.sigmoid(extreme_logits)
print("\nextreme case - sigmoid output:", extreme_probs.item(), "(rounds to exactly 0.0!)")

bce_with_logits_result = nn.BCEWithLogitsLoss()(extreme_logits, extreme_target)
print("BCEWithLogitsLoss handles it fine:", bce_with_logits_result.item())
# BCELoss(extreme_probs, extreme_target) would produce inf here - BCEWithLogitsLoss
# uses a numerically stable formula internally that avoids ever computing log(0)


BCELoss (manual sigmoid first): 0.29423221945762634
BCEWithLogitsLoss (raw logits): 0.29423215985298157
same result in the normal case: True

extreme case - sigmoid output: 0.0 (rounds to exactly 0.0!)
BCEWithLogitsLoss handles it fine: 100.0


### 7. Learning rate schedulers — StepLR and CosineAnnealingLR

**What this cell does (plain language):** we attach two different
schedulers to an optimizer and print the learning rate at each epoch, so you
can see exactly how each decay pattern looks in practice.


In [ ]:
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
step_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

print("StepLR (halve every 3 epochs):")
for epoch in range(10):
    print(f"  epoch {epoch}: lr = {optimizer.param_groups[0]['lr']:.5f}")
    optimizer.step()  # normally you'd do a real training step here
    step_scheduler.step()

model2 = nn.Linear(1, 1)
optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.1)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=10)

print("\nCosineAnnealingLR (smooth decay over 10 epochs):")
for epoch in range(10):
    print(f"  epoch {epoch}: lr = {optimizer2.param_groups[0]['lr']:.5f}")
    optimizer2.step()
    cosine_scheduler.step()


StepLR (halve every 3 epochs):
  epoch 0: lr = 0.10000
  epoch 1: lr = 0.10000
  epoch 2: lr = 0.10000
  epoch 3: lr = 0.05000
  epoch 4: lr = 0.05000
  epoch 5: lr = 0.05000
  epoch 6: lr = 0.02500
  epoch 7: lr = 0.02500
  epoch 8: lr = 0.02500
  epoch 9: lr = 0.01250

CosineAnnealingLR (smooth decay over 10 epochs):
  epoch 0: lr = 0.10000
  epoch 1: lr = 0.09755
  epoch 2: lr = 0.09045
  epoch 3: lr = 0.07939
  epoch 4: lr = 0.06545
  epoch 5: lr = 0.05000
  epoch 6: lr = 0.03455
  epoch 7: lr = 0.02061
  epoch 8: lr = 0.00955
  epoch 9: lr = 0.00245


### 8. ReduceLROnPlateau — reacting to validation loss

**What this cell does (plain language):** we simulate a validation loss that
plateaus (stops improving) partway through training, feed it to
`ReduceLROnPlateau` each epoch, and watch the learning rate drop
automatically once the plateau is detected.


In [8]:
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

# simulated validation losses: improving, then plateauing
fake_val_losses = [1.0, 0.8, 0.6, 0.59, 0.585, 0.584, 0.583, 0.3, 0.29]

for epoch, val_loss in enumerate(fake_val_losses):
    current_lr = optimizer.param_groups[0]['lr']
    print(f"epoch {epoch}: val_loss={val_loss:.3f}, lr={current_lr:.5f}")
    plateau_scheduler.step(val_loss)  # this scheduler needs the metric passed in

# notice the lr drops after the loss stalls around epochs 3-6, then may drop
# again if it plateaus further - patience=2 means it waits 2 epochs of no
# improvement before reducing


epoch 0: val_loss=1.000, lr=0.10000
epoch 1: val_loss=0.800, lr=0.10000
epoch 2: val_loss=0.600, lr=0.10000
epoch 3: val_loss=0.590, lr=0.10000
epoch 4: val_loss=0.585, lr=0.10000
epoch 5: val_loss=0.584, lr=0.10000
epoch 6: val_loss=0.583, lr=0.10000
epoch 7: val_loss=0.300, lr=0.10000
epoch 8: val_loss=0.290, lr=0.10000


### 9. Gradient clipping — preventing exploding gradients

**What this cell does (plain language):** we deliberately create a situation
with a very large gradient, then apply `clip_grad_norm_` and show the
gradient norm gets capped at the specified maximum, protecting the training
step from taking an enormous, destabilizing update.


In [9]:
model = nn.Linear(1, 1)
# manually set a huge gradient to simulate an exploding-gradient situation
model.weight.grad = torch.tensor([[1000.0]])
model.bias.grad = torch.tensor([500.0])

def total_grad_norm(model):
    return torch.sqrt(sum(p.grad.pow(2).sum() for p in model.parameters() if p.grad is not None))

print("gradient norm BEFORE clipping:", total_grad_norm(model).item())

torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

print("gradient norm AFTER clipping (should be ~1.0):", total_grad_norm(model).item())
print("\nclip_grad_norm_ rescales ALL gradients proportionally, preserving their")
print("relative direction while capping the combined magnitude")


gradient norm BEFORE clipping: 1118.033935546875
gradient norm AFTER clipping (should be ~1.0): 1.0

clip_grad_norm_ rescales ALL gradients proportionally, preserving their
relative direction while capping the combined magnitude


### 10. Comparing multiple optimizers side by side on one problem

**What this cell does (plain language):** we train the same small model with
four different optimizer configurations on identical data and print the
final loss for each, so you get an at-a-glance comparison of typical
behavior — useful intuition-building before you pick one for a real project.


In [10]:
optimizers_to_compare = {
    "SGD (lr=0.01)": lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD+momentum (lr=0.01, m=0.9)": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam (lr=0.01)": lambda p: torch.optim.Adam(p, lr=0.01),
    "AdamW (lr=0.01, wd=0.01)": lambda p: torch.optim.AdamW(p, lr=0.01, weight_decay=0.01),
}

print(f"{'Optimizer':35s} | Final loss (50 steps)")
print("-" * 60)
for name, opt_fn in optimizers_to_compare.items():
    losses = train(opt_fn, steps=50)
    print(f"{name:35s} | {losses[-1]:.5f}")


Optimizer                           | Final loss (50 steps)
------------------------------------------------------------
SGD (lr=0.01)                       | 0.00626
SGD+momentum (lr=0.01, m=0.9)       | 0.00357
Adam (lr=0.01)                      | 3.55948
AdamW (lr=0.01, wd=0.01)            | 3.62121


## Common bugs

- **Forgot `optimizer.zero_grad()`**
  Gradients accumulate silently across steps, effectively making the
  learning rate act like it's growing every iteration; training becomes
  unstable or diverges after a few steps. Fix: always call
  `optimizer.zero_grad()` before `loss.backward()` each iteration (unless
  you're deliberately doing gradient accumulation, see the autograd module).

- **Passing softmax output to `CrossEntropyLoss`**
  `nn.CrossEntropyLoss` applies `log_softmax` internally — passing
  already-softmaxed probabilities double-applies it and silently gives a
  wrong (usually much smaller, misleadingly "good-looking") loss value. Fix:
  pass raw logits directly, never apply softmax yourself before this loss.

- **Passing one-hot labels instead of integer class indices to `CrossEntropyLoss`**
  `RuntimeError` about size mismatch, or (worse) silently wrong results if
  shapes coincidentally align. Fix: labels should be a 1D tensor of integer
  class indices (`torch.long`), shape `(batch,)`, not one-hot `(batch, classes)`.

- **Using `BCELoss` on raw logits (without a sigmoid first)**
  Produces nonsensical loss values (or NaN) since `BCELoss` expects
  probabilities in `[0, 1]`, not arbitrary real-valued logits. Fix: either
  apply `torch.sigmoid()` first and use `BCELoss`, or (preferred) use
  `BCEWithLogitsLoss` directly on the raw logits.

- **Learning rate too high → loss becomes NaN almost immediately**
  Classic sign of an LR that's simply too large for the problem/optimizer
  combination. Fix: lower the LR by 10x and retry; consider Adam/AdamW which
  are more forgiving of LR choice than plain SGD; add gradient clipping.

- **Scheduler `.step()` called at the wrong frequency**
  Most schedulers (`StepLR`, `CosineAnnealingLR`) expect `.step()` once per
  EPOCH; a few (`OneCycleLR`) expect it once per BATCH. Calling at the wrong
  frequency silently gives a very different LR schedule than intended. Fix:
  check the specific scheduler's documented convention before wiring it in.

- **`ReduceLROnPlateau.step()` called without the metric argument**
  Unlike most schedulers, `ReduceLROnPlateau.step(val_loss)` needs the
  monitored metric passed in every call — calling plain `.step()` raises a
  `TypeError`. Fix: always pass the current validation metric:
  `scheduler.step(val_loss)`.

- **Using `weight_decay` with plain `Adam` instead of `AdamW`**
  Not an error, but a subtly incorrect regularization — Adam folds weight
  decay into the gradient/moment estimates rather than applying it directly
  to the weights, which behaves differently (and generally worse) than
  intended L2 regularization. Fix: use `AdamW` whenever you want weight decay.

- **Clipping gradients in the wrong order relative to backward()/step()**
  `clip_grad_norm_` must be called AFTER `loss.backward()` (gradients must
  exist to be clipped) and BEFORE `optimizer.step()` (so the clipped values
  are what actually get used in the update). Calling it at the wrong point
  either does nothing useful or clips already-applied updates too late.


## Exercises

Attempt each one in the empty cell right after the question. My full solution is in the cell after that - don't peek until you've tried.

**Exercise 1 - Build the full training cycle.**
Train an `nn.Linear(1, 1)` model to fit `y = 5x - 2` using SGD with
`lr=0.05` for 200 steps. Print the loss every 50 steps and the final learned
weight/bias.

In [ ]:
# Your attempt for Exercise 1 here\n

**Solution 1**

In [12]:
# Solution 1
torch.manual_seed(0)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [8.0], [13.0], [18.0]])  # y = 5x - 2

for step in range(200):
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(f"step {step}: loss = {loss.item():.4f}")

print("\nfinal weight (expected ~5.0):", model.weight.item())
print("final bias (expected ~-2.0):", model.bias.item())


step 0: loss = 130.9894
step 50: loss = 0.5337
step 100: loss = 0.1181
step 150: loss = 0.0261

final weight (expected ~5.0): 4.936707496643066
final bias (expected ~-2.0): -1.813912272453308


**Exercise 2 - Compare LR values.**
Train the same tiny model 3 times with SGD at `lr=0.001`, `lr=0.05`, and
`lr=0.5`. Print the final loss for each and explain in a comment what
happened with the highest learning rate.

In [13]:
# Your attempt for Exercise 2 here\n

**Solution 2**

In [14]:
# Solution 2
def quick_train(lr, steps=100):
    torch.manual_seed(0)
    model = nn.Linear(1, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
    y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
    return loss.item()

for lr in [0.001, 0.05, 0.5]:
    final_loss = quick_train(lr)
    print(f"lr={lr}: final loss = {final_loss}")

# With lr=0.5 the loss likely diverges to a very large number or becomes NaN -
# the learning rate is too high for this problem, causing updates to overshoot
# the minimum repeatedly and amplify instead of converge.


lr=0.001: final loss = 1.2561094760894775
lr=0.05: final loss = 0.00032546790316700935
lr=0.5: final loss = nan


**Exercise 3 - Fix the CrossEntropyLoss bug.**
This code computes an incorrect loss - find the bug and fix it:
```python
logits = torch.tensor([[3.0, 1.0, 0.2]])
probs = torch.softmax(logits, dim=1)
labels = torch.tensor([0])
loss = nn.CrossEntropyLoss()(probs, labels)
```

In [ ]:
# Your attempt for Exercise 3 here\n

**Solution 3**

In [16]:
# Solution 3
# Bug: softmax was applied manually before passing to CrossEntropyLoss,
# which applies its own internal log_softmax - this double-applies softmax
# and gives an incorrect loss value.

logits = torch.tensor([[3.0, 1.0, 0.2]])
labels = torch.tensor([0])
loss = nn.CrossEntropyLoss()(logits, labels)  # fix: pass raw logits directly
print("correct loss:", loss.item())


correct loss: 0.17910423874855042


**Exercise 4 - Choose the right loss function.**
For each scenario, name the correct PyTorch loss function and explain why
in one sentence: (a) predicting house prices (continuous value), (b)
classifying images into 10 categories, (c) predicting whether an email is
spam or not spam (single binary output).

In [ ]:
# Your attempt for Exercise 4 here\n

**Solution 4**

In [ ]:
# Solution 4
# (a) house prices - nn.MSELoss (or nn.L1Loss/SmoothL1Loss if outliers are a
#     concern) - it's a regression problem with a continuous target value.
# (b) 10-category image classification - nn.CrossEntropyLoss - standard for
#     multi-class classification with mutually exclusive classes, works
#     directly on raw logits.
# (c) spam / not spam - nn.BCEWithLogitsLoss - binary classification with a
#     single output logit; this is numerically more stable than sigmoid + BCELoss.
print("see comments above for the reasoning")


see comments above for the reasoning


**Exercise 5 - Implement gradient clipping in a loop.**
Write a training loop for any small model that includes gradient clipping
with `max_norm=1.0`, placed correctly relative to `backward()` and `step()`.

In [ ]:
# Your attempt for Exercise 5 here\n

**Solution 5**

In [ ]:
# Solution 5
model = nn.Linear(10, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

x = torch.randn(8, 10)
y = torch.randn(8, 1)

optimizer.zero_grad()
loss = loss_fn(model(x), y)
loss.backward()                                              # 1. compute gradients
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 2. clip AFTER backward
optimizer.step()                                              # 3. update AFTER clipping

print("training step with gradient clipping completed")


training step with gradient clipping completed


**Exercise 6 - StepLR by hand.**
Given `StepLR(optimizer, step_size=5, gamma=0.1)` starting at `lr=1.0`, what
will the learning rate be at epoch 12? Compute it by hand, then verify with code.

In [ ]:
# Your attempt for Exercise 6 here\n

**Solution 6**

In [ ]:
# Solution 6
# By hand: StepLR halves... no wait, gamma=0.1 multiplies lr by 0.1 every
# step_size=5 epochs. Starting lr=1.0:
# epochs 0-4: lr=1.0
# epochs 5-9: lr=0.1 (after 1 decay)
# epochs 10-14: lr=0.01 (after 2 decays)
# so at epoch 12: lr = 1.0 * 0.1^2 = 0.01

model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

for epoch in range(13):
    if epoch == 12:
        print(f"epoch {epoch}: lr = {optimizer.param_groups[0]['lr']} (expected 0.01)")
    optimizer.step()
    scheduler.step()


epoch 12: lr = 0.010000000000000002 (expected 0.01)


**Exercise 7 - Weight decay comparison.**
Train two identical models, one with `Adam(weight_decay=0.5)` (a
deliberately large value) and one with `AdamW(weight_decay=0.5)`, for 30
steps. Print the final weights and comment on why they differ.

In [ ]:
# Your attempt for Exercise 7 here\n

**Solution 7**

In [24]:
# Solution 7
torch.manual_seed(0)
model_adam = nn.Linear(1, 1)
torch.manual_seed(0)
model_adamw = nn.Linear(1, 1)

opt_adam = torch.optim.Adam(model_adam.parameters(), lr=0.1, weight_decay=0.5)
opt_adamw = torch.optim.AdamW(model_adamw.parameters(), lr=0.1, weight_decay=0.5)

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])
loss_fn = nn.MSELoss()

for _ in range(30):
    for model, opt in [(model_adam, opt_adam), (model_adamw, opt_adamw)]:
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()

print("Adam final weight:", model_adam.weight.item())
print("AdamW final weight:", model_adamw.weight.item())
# They differ noticeably because AdamW applies the (large) weight decay
# directly to the weights each step, while Adam mixes it into the gradient
# before the adaptive moment estimates are computed - a fundamentally
# different (and for Adam, less correct) form of regularization.


Adam final weight: 1.8732341527938843
AdamW final weight: 1.2586408853530884


**Exercise 8 - ReduceLROnPlateau wiring.**
Write a training loop skeleton (pseudocode/comments are fine for the actual
training part) that correctly evaluates on a validation set each epoch and
feeds the validation loss into a `ReduceLROnPlateau` scheduler.

In [ ]:
# Your attempt for Exercise 8 here\n

**Solution 8**

In [ ]:
# Solution 8
# Pseudocode/skeleton - focuses on the scheduler wiring, not full training details

model = nn.Linear(10, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
loss_fn = nn.MSELoss()

# for epoch in range(num_epochs):
#     model.train()
#     for x_batch, y_batch in train_loader:
#         optimizer.zero_grad()
#         loss = loss_fn(model(x_batch), y_batch)
#         loss.backward()
#         optimizer.step()
#
#     model.eval()
#     val_losses = []
#     with torch.no_grad():
#         for x_val, y_val in val_loader:
#             val_losses.append(loss_fn(model(x_val), y_val).item())
#     avg_val_loss = sum(val_losses) / len(val_losses)
#
#     scheduler.step(avg_val_loss)  # <- the key line: pass the METRIC, not just call .step()

print("skeleton defined - see comments for the full loop structure")


skeleton defined - see comments for the full loop structure


**Exercise 9 - BCEWithLogitsLoss with class imbalance.**
`nn.BCEWithLogitsLoss` has a `pos_weight` argument for handling class
imbalance. Given a dataset with 9x more negative than positive examples,
compute the appropriate `pos_weight` value and use it in a loss calculation.

In [ ]:
# Your attempt for Exercise 9 here\n

**Solution 9**

In [ ]:
# Solution 9
# pos_weight should roughly equal (number of negatives / number of positives)
# to upweight the rare positive class in the loss.
num_negative = 900
num_positive = 100
pos_weight_value = num_negative / num_positive  # = 9.0

pos_weight = torch.tensor([pos_weight_value])
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

logits = torch.tensor([2.0, -1.0, 0.5])
targets = torch.tensor([1.0, 0.0, 1.0])
loss = loss_fn(logits, targets)
print("pos_weight used:", pos_weight_value)
print("loss with pos_weight:", loss.item())


pos_weight used: 9.0
loss with pos_weight: 1.9074357748031616


**Exercise 10 - Optimizer state inspection.**
After a few training steps with Adam, write code to inspect the optimizer's
internal state for one parameter (hint: `optimizer.state`) and print the
keys available there (they represent Adam's internal moment estimates).

In [ ]:
# Your attempt for Exercise 10 here\n

**Solution 10**

In [ ]:
# Solution 10
model = nn.Linear(5, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

x = torch.randn(4, 5)
y = torch.randn(4, 1)

for _ in range(3):
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()

# inspect optimizer state for the weight parameter
weight_param = model.weight
state = optimizer.state[weight_param]
print("keys in Adam's internal state for this parameter:", list(state.keys()))
# typically: 'step', 'exp_avg' (first moment / momentum-like term),
# 'exp_avg_sq' (second moment / adaptive scaling term)


keys in Adam's internal state for this parameter: ['step', 'exp_avg', 'exp_avg_sq']
